# Bangla Hallucination Detection - Model Pipeline (v2.0)

This notebook trains a Bangla LLM Hallucination Detection system designed to run on Kaggle.
It implements an NLI (Natural Language Inference) Entailment score module combined with TF-IDF character-level classification. 

### Version History
- **v1.0**: Initial baseline version using character-level n-gram TF-IDF on `prompt_bn` and `response_bn`.
- **v2.0 (Current)**: Incorporates NLI zero-shot entailment scores using `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` for context-grounded prediction, fallback classifier for closed-book query items, and hybrid rules.

In [ ]:
import json
import numpy as np
import pandas as pd
import os
import torch
from tqdm.auto import tqdm
from transformers import pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

RANDOM_STATE = 42
DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SUBMISSION_PATH = "submission.csv"

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load and Clean Data

In [ ]:
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../bengali-hallucination/dataset samples.json"
    TEST_PATH = "../bengali-hallucination/test set.csv"

with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"{len(df)} training rows loaded.")

for col in ["prompt_bn", "response_bn"]:
    df[col] = df[col].astype(str)

NO_CONTEXT_VALUES = {"", "nan", "NaN", "[NULL]", None}

def clean_context(value):
    if pd.isna(value) or str(value).strip() in NO_CONTEXT_VALUES:
        return ""
    return str(value).strip()

df["context"] = df["context"].apply(clean_context)
df["has_context"] = df["context"].str.len() > 0

## 2. Load NLI Model Pipeline

In [ ]:
print(f"Loading Zero-Shot Multilingual NLI Pipeline: {NLI_MODEL}")
nli_pipe = pipeline(
    'zero-shot-classification',
    model=NLI_MODEL,
    device=0 if device == 'cuda' else -1,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
print("NLI model pipeline ready.")

## 3. Define Fallback Classifier for Closed-Book Rows

In [ ]:
def select_column(name):
    return FunctionTransformer(lambda frame: frame[name].to_numpy(), validate=False)

def tfidf_branch(name):
    return Pipeline([
        ("select", select_column(name)),
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=20000)),
    ])

baseline_clf = Pipeline([
    ("features", FeatureUnion([
        ("prompt", tfidf_branch("prompt_bn")),
        ("response", tfidf_branch("response_bn")),
    ])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

## 4. Run Inference/Predictions with Hybrid Logic

In [ ]:
def run_hybrid_inference(data_df, train_labels=None):
    """
    For rows with context: compute NLI entailment score.
    For rows without context: fallback to TF-IDF classifier predictions.
    """
    preds = []
    
    # Train fallback classifier if labels are provided
    if train_labels is not None:
        baseline_clf.fit(data_df, train_labels)
        
    fallback_preds = baseline_clf.predict(data_df)
    
    print("Running Hybrid Inference...")
    for i, row in tqdm(enumerate(data_df.itertuples()), total=len(data_df)):
        if row.has_context:
            # Compute NLI score
            try:
                res = nli_pipe(
                    row.context,
                    candidate_labels=[row.response_bn],
                    hypothesis_template="{}",
                    multi_label=True,
                )
                score = res["scores"][0]
                pred = 1 if score > 0.5 else 0
            except Exception as e:
                print(f"NLI failed for row {i}: {e}. Falling back to TF-IDF.")
                pred = fallback_preds[i]
        else:
            # Closed book queries fallback to TF-IDF
            pred = fallback_preds[i]
            
        preds.append(pred)
        
    return np.array(preds)

## 5. Evaluate Hybrid Pipeline

In [ ]:
y = df["label"].values
# Run model evaluation (simulating validation loop using the training set / samples)
cv_preds = run_hybrid_inference(df, train_labels=y)

print("=== v2.0 Hybrid NLI & TF-IDF Model Evaluation Metrics ===")
print(f"Accuracy : {accuracy_score(y, cv_preds):.4f}")
print(f"Macro-F1 : {f1_score(y, cv_preds, average='macro'):.4f} (Primary Metric)")
print(f"F1 (label=1 - faithful): {f1_score(y, cv_preds, pos_label=1):.4f}")
print(f"F1 (label=0 - hallucinated): {f1_score(y, cv_preds, pos_label=0):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y, cv_preds))

## 6. Generate Submission File

In [ ]:
if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    print(f"Loaded test set with {len(test_df)} rows.")
    for col in ["prompt_bn", "response_bn"]:
        test_df[col] = test_df[col].astype(str)
    test_df["context"] = test_df["context"].apply(clean_context)
    test_df["has_context"] = test_df["context"].str.len() > 0
    
    test_preds = run_hybrid_inference(test_df, train_labels=None) # fallback_clf already fit on full train set in step 5
    
    submission = pd.DataFrame({
        "id": test_df["id"] if "id" in test_df.columns else range(len(test_df)),
        "label": test_preds
    })
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Submission file saved to {SUBMISSION_PATH}. Rows: {len(submission)}")
else:
    print("Test set CSV file not found. Skipping prediction. Ready for Kaggle run.")